In [ ]:
# for time series manipulation
import pandas

class DataDownloader:
    def download_time_series(self, vendor_ticker, pretty_ticker, start_date, source, csv_file = None):

        if source == 'Quandl':
            import Quandl
            # Quandl requires API key for large number of daily downloads
            # https://www.quandl.com/help/api
            spot = Quandl.get(vendor_ticker)    # Bank of England's database on Quandl
            spot = pandas.DataFrame(data=spot['Value'], index=spot.index)
            spot.columns = [pretty_ticker]

        elif source == 'Bloomberg':
            from bbg_com import HistoricalDataRequest
            req = HistoricalDataRequest([vendor_ticker], ['PX_LAST'], start = start_date)
            req.execute()

            spot = req.response_as_single()
            spot.columns = [pretty_ticker]
        elif source == 'CSV':
            dateparse = lambda x: pandas.datetime.strptime(x, '%Y-%m-%d')

            # in case you want to use a source other than Bloomberg/Quandl
            spot = pandas.read_csv(csv_file, index_col=0, parse_dates=0, date_parser=dateparse)

        return spot

In [ ]:
# for dates
import datetime

# time series manipulation
import pandas

# for plotting data
import plotly
from plotly.graph_objs import *

class EventPlot:
    def event_study(self, spot, dates, pre, post, mean_label = 'Mean'):
        # event_study - calculates the asset price moves over windows around event days
        #
        # spot = price of asset to study
        # dates = event days to anchor our event study
        # pre = days before the event day to start our study
        # post = days after the event day to start our study
        #
        data_frame = pandas.DataFrame()

        # for each date grab spot data the days before and after
        for i in range(0, len(dates)):
            mid_index = spot.index.searchsorted(dates[i])
            start_index = mid_index + pre
            finish_index = mid_index + post
            spot.reset_index(inplace=True)
            x = (spot.loc[start_index:finish_index])[spot.columns.values[0]]

            data_frame[dates[i]] = x.values
            print(x.values)
        print(1)
        data_frame.index = range(pre, post + 1)
        print(2)

        data_frame = data_frame / data_frame.shift(1) - 1   # returns
        print(3)

        # add the mean on to the end
        data_frame[mean_label] = data_frame.mean(axis=1)
        print(4)

        data_frame = 100.0 * (1.0 + data_frame).cumprod()   # index
        data_frame.loc[pre,:] = 100

        return data_frame

In [ ]:
def parse_dates(self, str_dates):
    # parse_dates - parses string dates into Python format
    #
    # str_dates = dates to be parsed in the format of day/month/year
    #

    dates = []

    for d in str_dates:
        dates.append(datetime.datetime.strptime(d, '%d/%m/%Y'))

    return dates

EventPlot.parse_dates = parse_dates

In [ ]:
def create_layout(self, title, xaxis, yaxis, width = -1, height = -1):
    # create_layout - populates a layout object
    # title = title of the plot
    # xaxis = xaxis label
    # yaxis = yaxis label
    # width (optional) = width of plot
    # height (optional) = height of plot
    #

    layout = Layout(
                title = title,
                xaxis = plotly.graph_objs.XAxis(
                    title = xaxis,
                    showgrid = False
            ),
                yaxis = plotly.graph_objs.YAxis(
                    title= yaxis,
                    showline = False
            )
        )

    if width > 0 and height > 0:
        layout['width'] = width
        layout['height'] = height

    return layout

EventPlot.create_layout = create_layout

In [ ]:
def convert_df_plotly(self, dataframe, axis_no = 1, color_def = ['default'],
                        special_line = 'Mean', showlegend = True, addmarker = False, gradcolor = None):
    # convert_df_plotly - converts a Pandas data frame to Plotly format for line plots
    # dataframe = data frame due to be converted
    # axis_no = axis for plot to be drawn (default = 1)
    # special_line = make lines named this extra thick
    # color_def = color scheme to be used (default = ['default']), colour will alternate in the list
    # showlegend = True or False to show legend of this line on plot
    # addmarker = True or False to add markers
    # gradcolor = Create a graduated color scheme for the lines
    #
    # Also see http://nbviewer.ipython.org/gist/nipunreddevil/7734529 for converting dataframe to traces
    # Also see http://moderndata.plot.ly/color-scales-in-ipython-notebook/

    x = dataframe.index.values

    traces = []

    # will be used for market opacity for the markers
    increments = 0.95 / float(len(dataframe.columns))

    if gradcolor is not None:
        try:
            import colorlover as cl
            color_def = cl.scales[str(len(dataframe.columns))]['seq'][gradcolor]
        except:
            print('Check colorlover installation...')

    i = 0

    for key in dataframe:
        scatter = plotly.graph_objs.Scatter(
                    x = x,
                    y = dataframe[key].values,
                    name = key,
                    xaxis = 'x' + str(axis_no),
                    yaxis = 'y' + str(axis_no),
                    showlegend = showlegend)

        # only apply color/marker properties if not "default"
        if color_def[i % len(color_def)] != "default":
            if special_line in str(key):
                # special case for lines labelled "mean"
                # make line thicker
                scatter['mode'] = 'lines'
                scatter['line'] = plotly.graph_objs.Line(
                            color = color_def[i % len(color_def)],
                            width = 2
                        )
            else:
                line_width = 1

                # set properties for the markers which change opacity
                # for markers make lines thinner
                if addmarker:
                    opacity = 0.05 + (increments * i)
                    scatter['mode'] = 'markers+lines'
                    scatter['marker'] = plotly.graph_objs.Marker(
                                color=color_def[i % len(color_def)],  # marker color
                                opacity = opacity,
                                size = 5)
                    line_width = 0.2

                else:
                    scatter['mode'] = 'lines'

                scatter['line'] = plotly.graph_objs.Line(
                        color = color_def[i % len(color_def)],
                        width = line_width)

            i = i + 1

        traces.append(scatter)

    return traces

EventPlot.convert_df_plotly = convert_df_plotly

# UK ELECTION

In [ ]:
# for time series/maths
import pandas

# for plotting data
import chart_studio.plotly
import chart_studio.plotly.plotly as py
from plotly.graph_objs import *

def ukelection():
    # Learn about API authentication here: https://plotly.com/python/getting-started
    # Find your api_key here: https://plotly.com/settings/api
    plotly_username = "thalesians"
    plotly_api_key = "XXXXXXXXX"

    plotly.tools.set_credentials_file(username=plotly_username, api_key=plotly_api_key)

In [ ]:
ticker = 'GBPUSD' # will use in plot titles later (and for creating Plotly URL)

##### download market GBP/USD data from Quandl, Bloomberg or CSV file
source = "Bloomberg"
# source  = "Quandl"
# source = "CSV"

csv_file = None

event_plot = EventPlot()

data_downloader = DataDownloader()
start_date = event_plot.parse_dates(['01/01/1975'])

if source == 'Quandl':
    vendor_ticker = "BOE/XUDLUSS"
elif source == 'Bloomberg':
    vendor_ticker = 'GBPUSD BGN Curncy'
elif source == 'CSV':
    vendor_ticker = 'GBPUSD'
    csv_file = 'D:/GBPUSD.csv'

import pandas as pd
# spot = data_downloader.download_time_series(vendor_ticker, ticker, start_date[0], source, csv_file = csv_file)
spot = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfDaily.pkl")


In [ ]:
spot

In [ ]:
labour_wins = ['18/03/2021']#, '10/11/2021', '05/05/2021', '07/06/2021', '05/05/2021']
conservative_wins = ['03/05/2021']#, '09/06/2021', '11/06/2021', '09/04/2021', '06/05/2021']

# convert to more easily readable format
labour_wins_d = event_plot.parse_dates(labour_wins)
conservative_wins_d = event_plot.parse_dates(conservative_wins)

# only takes those elections where we have data
labour_wins_d = [d for d in labour_wins_d if d > spot.index[0].to_pydatetime()]
conservative_wins_d = [d for d in conservative_wins_d if d > spot.index[0].to_pydatetime()]

spot.index.name = 'Date'

In [ ]:
spot.index = spot.Date
spot = spot[["VIX","Sentiment"]]

In [ ]:
# number of days before and after for our event study
pre = -20
post = 20

# calculate spot path during Labour wins
labour_wins_spot = event_plot.event_study(spot, labour_wins_d, pre, post, mean_label = 'Labour Mean')

# calculate spot path during Conservative wins
conservative_wins_spot = event_plot.event_study(spot, conservative_wins_d, pre, post, mean_label = 'Conservative Mean')

In [ ]:
spot

In [ ]:
##### Create separate plots of price action during Labour and Conservative wins
xaxis = 'Days'
yaxis = 'Index'
source_label = "Source: @thalesians/BBG/Wikipedia"

In [ ]:
###### Plot market reaction during Labour UK election wins
###### Using default color scheme

title = ticker + ' during UK gen elect - Lab wins' + '<BR>' + source_label

fig = Figure(data=event_plot.convert_df_plotly(labour_wins_spot),
                layout=event_plot.create_layout(title, xaxis, yaxis)
)

py.iplot(fig, filename='labour-wins-' + ticker)